# Restoration Fusion: Colab backend
Runs three frozen pretrained restorers (HINet, SFNet, AdaIR) plus our fusion module
behind a small FastAPI server, and exposes it through a public tunnel.
The React frontend only sends one image and shows the four outputs.

**Run the cells top to bottom on a GPU runtime (T4 is enough).**
Cells marked CONFIG are the ones you may need to edit.

## 1. Setup

In [ ]:
import os, sys, re, json, glob, io, time, base64, shutil, subprocess, threading
import torch
assert torch.cuda.is_available(), "Switch to a GPU runtime: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "einops", "fastapi", "uvicorn",
                "python-multipart", "scikit-image", "lmdb", "pyyaml", "gdown", "huggingface_hub",
                "gguf", "pytorch_msssim", "opencv-python-headless"], check=True)

## 2. CONFIG: weights and repos
* HINet and AdaIR weights are fetched automatically (same sources as your evaluation notebooks).
* SFNet weights come from the authors' Drive folder. If auto-download does not find the right
  file, download it by hand, upload it to Colab and set the path below.
* Fusion checkpoints (`fusion_deblur.pth`, `fusion_derain.pth`) are optional. Without them the
  fusion module is an identity-initialised placeholder that behaves like a plain average of the
  available models, and the API reports `fusion_trained: false`.

In [ ]:
WORK = "/content"
WEIGHTS = f"{WORK}/weights"; os.makedirs(WEIGHTS, exist_ok=True)

# --- SFNet checkpoints: set a path to override auto-discovery ---
SFNET_DEBLUR_CKPT = ""     # e.g. "/content/weights/sfnet/GoPro.pkl"
SFNET_DERAIN_CKPT = ""     # e.g. "/content/weights/sfnet/Rain.pkl"  (leave "" if you have none)
SFNET_DRIVE_FOLDER = "1OJv9d6e90hlpDSyo8oJY-END3xj4nUmg"   # GoPro/RSBlur weights folder from the SFNet README
SFNET_DERAIN_DRIVE_FOLDER = ""   # paste the Drive folder id from SFNet/Image_deraining README if it has one

# --- AdaIR: direct PyTorch checkpoint URL (preferred) or the HF GGUF conversion used in your notebook ---
ADAIR_CKPT_URL = ""
ADAIR_GGUF_REPO = "cstr/adair-GGUF"

# --- Fusion checkpoints (state_dict .pth). Save them here after training. ---
FUSION_CKPT = {"deblur": f"{WEIGHTS}/fusion_deblur.pth", "derain": f"{WEIGHTS}/fusion_derain.pth"}

# --- Serving ---
MAX_SIDE = 1024          # longest image side processed (bigger inputs are downscaled to fit a T4)
PORT = 8000

In [ ]:
def sh(cmd):
    print("$", cmd); return subprocess.run(cmd, shell=True, check=False)

for name, url in {"HINet": "https://github.com/megvii-model/HINet.git",
                  "SFNet": "https://github.com/c-yn/SFNet.git",
                  "AdaIR": "https://github.com/c-yn/AdaIR.git"}.items():
    if not os.path.isdir(f"{WORK}/{name}"):
        sh(f"git clone --depth 1 {url} {WORK}/{name}")

# basicsr/version.py is normally generated by setup.py (same workaround as your HINet notebook)
vp = f"{WORK}/HINet/basicsr/version.py"
if not os.path.exists(vp):
    open(vp, "w").write("__version__ = '1.2.0'\nshort_version = '1.2.0'\nversion_info = (1, 2, 0)\n")

# HINet official checkpoints
HINET_CKPT = {"deblur": f"{WEIGHTS}/HINet-GoPro.pth", "derain": f"{WEIGHTS}/HINet-Rain13k.pth"}
for path, gid in [(HINET_CKPT["deblur"], "1dw8PKVkLfISzNtUu3gqGh83NBO83ZQ5n"),
                  (HINET_CKPT["derain"], "1AVedAkb1B2F2b3XGWlMFFVSsNfQlCwxa")]:
    if not (os.path.exists(path) and os.path.getsize(path) > 1_000_000):
        sh(f'gdown "https://drive.google.com/uc?id={gid}" -O {path}')

# SFNet weights (best effort: grab the Drive folder, then look for a matching file)
def find_ckpt(folder, *words):
    cands = [p for p in glob.glob(f"{folder}/**/*", recursive=True)
             if p.lower().endswith((".pkl", ".pth", ".pt", ".ckpt")) and any(w in os.path.basename(p).lower() for w in words)]
    return sorted(cands)[0] if cands else ""

if SFNET_DRIVE_FOLDER and not glob.glob(f"{WEIGHTS}/sfnet/**/*.pkl", recursive=True):
    sh(f'gdown --folder "https://drive.google.com/drive/folders/{SFNET_DRIVE_FOLDER}" -O {WEIGHTS}/sfnet')
if SFNET_DERAIN_DRIVE_FOLDER and not glob.glob(f"{WEIGHTS}/sfnet_derain/**/*", recursive=True):
    sh(f'gdown --folder "https://drive.google.com/drive/folders/{SFNET_DERAIN_DRIVE_FOLDER}" -O {WEIGHTS}/sfnet_derain')
SFNET_DEBLUR_CKPT = SFNET_DEBLUR_CKPT or find_ckpt(f"{WEIGHTS}/sfnet", "gopro")
SFNET_DERAIN_CKPT = SFNET_DERAIN_CKPT or find_ckpt(f"{WEIGHTS}/sfnet_derain", "rain", "derain")
print("SFNet deblur ckpt:", SFNET_DEBLUR_CKPT or "NOT FOUND")
print("SFNet derain ckpt:", SFNET_DERAIN_CKPT or "NOT FOUND (SFNet will be skipped for deraining)")

# AdaIR weights
ADAIR_CKPT = ""
if ADAIR_CKPT_URL:
    ADAIR_CKPT = f"{WEIGHTS}/adair5d.ckpt"
    if not os.path.exists(ADAIR_CKPT):
        sh(f'wget -q "{ADAIR_CKPT_URL}" -O {ADAIR_CKPT}')
else:
    from huggingface_hub import hf_hub_download, list_repo_files
    ggufs = [f for f in list_repo_files(ADAIR_GGUF_REPO) if f.lower().endswith(".gguf")]
    chosen = ([f for f in ggufs if "f32" in f.lower()] or ggufs)[0]
    ADAIR_CKPT = hf_hub_download(ADAIR_GGUF_REPO, chosen, local_dir=WEIGHTS)
print("AdaIR ckpt:", ADAIR_CKPT)

## 3. Model wrappers
Every wrapper returns `(restored_image, feature_map)`.
The feature map is the input of the network's last 3-channel output convolution, i.e. the
decoder feature right before it is turned into pixels. If a network has several such layers, the one
that executes last wins (that is the full-resolution output head).

In [ ]:
import numpy as np, yaml
import torch.nn as nn, torch.nn.functional as F

dev = "cuda"
CLASH = {"utils", "data", "models", "net", "options", "train", "eval", "main", "layers", "losses"}

def isolated_import(repo_dir, fn):
    """Import from one repo without leaving its generic top-level names (utils, models, ...)
    in sys.modules, because HINet / SFNet / AdaIR all use such names."""
    before = set(sys.modules)
    sys.path.insert(0, repo_dir)
    try:
        return fn()
    finally:
        sys.path.remove(repo_dir)
        for m in list(sys.modules):
            if m not in before and m.split(".")[0] in CLASH:
                del sys.modules[m]

def strip_module(sd):
    return {(k[7:] if k.startswith("module.") else k): v for k, v in sd.items()}

class FeatureTap:
    """Records the input of the most recently executed Conv2d(out_channels=3)."""
    def __init__(self, net, layer_name=None):
        self.feat, self.layer = None, None
        mods = dict(net.named_modules())
        targets = {layer_name: mods[layer_name]} if layer_name else \
                  {n: m for n, m in mods.items() if isinstance(m, nn.Conv2d) and m.out_channels == 3}
        assert targets, "no 3-channel output conv found; pass layer_name"
        for n, m in targets.items():
            m.register_forward_pre_hook(lambda mod, inp, n=n: self._store(n, inp[0]))
        self.candidates = list(targets)
    def _store(self, name, x):
        self.feat, self.layer = x.detach(), name

class Restorer:
    def __init__(self, key, name, net, tap_layer=None):
        self.key, self.name = key, name
        self.net = net.to(dev).eval()
        for p in self.net.parameters(): p.requires_grad_(False)
        self.tap = FeatureTap(self.net, tap_layer)
        self.feat_channels = None
    @torch.inference_mode()
    def __call__(self, x):
        self.tap.feat = None
        out = self.net(x)
        if isinstance(out, (list, tuple)): out = out[-1]
        feat = self.tap.feat
        self.feat_channels = feat.shape[1]
        return out.clamp(0, 1), feat

# ---------- HINet (official repo, architecture parameters read from the repo's own test YAML) ----------
def load_hinet(task):
    def _imp():
        from basicsr.models.archs.hinet_arch import HINet
        return HINet
    HINet = isolated_import(f"{WORK}/HINet", _imp)
    yml = {"deblur": "options/test/GoPro/HINet-GoPro.yml", "derain": "options/test/Rain13k/HINet-Rain100L.yml"}[task]
    cfg = yaml.safe_load(open(f"{WORK}/HINet/{yml}"))["network_g"]; cfg.pop("type")
    net = HINet(**cfg)
    ck = torch.load(HINET_CKPT[task], map_location="cpu")
    net.load_state_dict(strip_module(ck.get("params", ck)), strict=True)
    return Restorer("hinet", "HINet", net)

# ---------- SFNet (official repo; one sub-folder per task) ----------
def load_sfnet(task):
    ckpt = {"deblur": SFNET_DEBLUR_CKPT, "derain": SFNET_DERAIN_CKPT}[task]
    if not ckpt: raise FileNotFoundError(f"no SFNet checkpoint configured for {task}")
    sub = {"deblur": "Motion_deblurring", "derain": "Image_deraining"}[task]
    def _imp():
        import importlib
        for modname in ("models.SFNet", "models.model", "model", "models"):
            try:
                mod = importlib.import_module(modname)
                if hasattr(mod, "build_net"): return mod.build_net
            except ImportError:
                continue
        raise ImportError(f"could not find build_net in {WORK}/SFNet/{sub}; open the folder and adapt load_sfnet")
    build_net = isolated_import(f"{WORK}/SFNet/{sub}", _imp)
    net = build_net()
    ck = torch.load(ckpt, map_location="cpu")
    net.load_state_dict(strip_module(ck.get("model", ck.get("params", ck.get("state_dict", ck)))), strict=True)
    return Restorer("sfnet", "SFNet", net)

# ---------- AdaIR (all-in-one checkpoint, used for both tasks) ----------
_ADAIR = {}
def load_adair(task):
    if "net" not in _ADAIR:
        AdaIR = isolated_import(f"{WORK}/AdaIR", lambda: __import__("net.model", fromlist=["AdaIR"]).AdaIR)
        net = AdaIR(decoder=True)
        target = net.state_dict()
        if ADAIR_CKPT.endswith(".gguf"):                       # same conversion logic as your AdaIR notebook
            from gguf import GGUFReader
            raw = {t.name: torch.from_numpy(np.array(t.data, dtype=np.float32, copy=True)) for t in GGUFReader(ADAIR_CKPT).tensors}
        else:
            ck = torch.load(ADAIR_CKPT, map_location="cpu")
            raw = ck.get("state_dict", ck)
        for p in ("net.", "module.net.", "module."):
            if raw and all(k.startswith(p) for k in raw):
                raw = {k[len(p):]: v for k, v in raw.items()}
        tk = list(target)
        if all(k in raw and raw[k].numel() == target[k].numel() for k in tk):
            mapped = {k: raw[k].reshape(target[k].shape).float() for k in tk}
        else:
            rv = list(raw.values()); assert len(rv) == len(tk), "AdaIR weight mapping failed (tensor count differs)"
            assert all(tuple(r.shape) == tuple(target[k].shape) for r, k in zip(rv, tk)), "AdaIR weight mapping failed (shape mismatch)"
            mapped = {k: r.float() for k, r in zip(tk, rv)}
        net.load_state_dict(mapped, strict=True)
        _ADAIR["net"] = net
    return Restorer("adair", "AdaIR", _ADAIR["net"])

LOADERS = {"hinet": load_hinet, "sfnet": load_sfnet, "adair": load_adair}
ORDER = ["hinet", "sfnet", "adair"]
MODELS = {"deblur": {}, "derain": {}}
LOAD_NOTES = {"deblur": {}, "derain": {}}
for task in MODELS:
    for key in ORDER:
        try:
            MODELS[task][key] = LOADERS[key](task)
            print(f"[ok]   {task:6s} {key}")
        except Exception as e:
            MODELS[task][key] = None
            LOAD_NOTES[task][key] = f"{type(e).__name__}: {str(e)[:200]}"
            print(f"[SKIP] {task:6s} {key}: {LOAD_NOTES[task][key]}")

## 4. Fusion module (our model, CMFNet-style, pixel-wise trust)
* **Align**: 1x1 conv per model maps its features to a common width; they are resized to image size.
* **Attention fusion**: channel attention picks useful features, then a head predicts one weight
  per model per pixel (softmax over models).
* **Image mix**: restored images are blended with those weights.
* **Refine**: a small residual correction is added.

Weight head and correction conv start at zero, so an untrained module equals a simple average.
This is a working skeleton, since the final design is not decided yet. Edit freely, but keep the
`forward(imgs, feats, mask)` signature so the server does not change.

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, c, r=4):
        super().__init__()
        self.f = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c, max(c // r, 4), 1), nn.PReLU(), nn.Conv2d(max(c // r, 4), c, 1), nn.Sigmoid())
    def forward(self, x): return x * self.f(x)

class FusionModule(nn.Module):
    def __init__(self, feat_channels=(64, 32, 48), dim=32):
        super().__init__()
        self.k = len(feat_channels)
        self.align = nn.ModuleList([nn.Conv2d(c, dim, 1) for c in feat_channels])
        self.fuse = nn.Sequential(nn.Conv2d(dim * self.k, dim, 3, padding=1), nn.PReLU(), ChannelAttention(dim))
        self.weight_head = nn.Conv2d(dim, self.k, 3, padding=1)
        self.refine = nn.Sequential(nn.Conv2d(dim, dim, 3, padding=1), nn.PReLU(), nn.Conv2d(dim, 3, 3, padding=1))
        for m in (self.weight_head, self.refine[-1]):
            nn.init.zeros_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, imgs, feats, mask):
        """imgs: K tensors [B,3,H,W]; feats: K tensors [B,C_k,h,w] (or None); mask: K bools (model available)."""
        H, W = imgs[0].shape[-2:]
        aligned = []
        for i, (f, ok) in enumerate(zip(feats, mask)):
            if ok and f is not None:
                a = self.align[i](f)
                if a.shape[-2:] != (H, W): a = F.interpolate(a, size=(H, W), mode="bilinear", align_corners=False)
            else:
                a = torch.zeros(imgs[0].shape[0], self.align[i].out_channels, H, W, device=imgs[0].device)
            aligned.append(a)
        fused = self.fuse(torch.cat(aligned, 1))
        logits = self.weight_head(fused)
        neg = torch.tensor([0.0 if ok else -1e4 for ok in mask], device=logits.device).view(1, -1, 1, 1)
        w = torch.softmax(logits + neg, dim=1)
        blend = sum(w[:, i:i + 1] * imgs[i] for i in range(self.k) if mask[i])
        return (blend + self.refine(fused)).clamp(0, 1), w

FUSION, FUSION_TRAINED = {}, {}
def init_fusion(task):
    # feature widths are discovered with a tiny dry run so the module matches the real taps
    x = torch.rand(1, 3, 64, 64, device=dev); chans = []
    for key, default in zip(ORDER, (64, 32, 48)):
        m = MODELS[task][key]
        if m:
            try: m(x); chans.append(m.feat_channels)
            except Exception as e: print("dry run failed for", task, key, e); chans.append(default)
        else: chans.append(default)
    net = FusionModule(tuple(chans)).to(dev).eval()
    path = FUSION_CKPT[task]
    if os.path.exists(path):
        ck = torch.load(path, map_location="cpu")
        net.load_state_dict(strip_module(ck.get("params", ck.get("model", ck))), strict=True)
        FUSION_TRAINED[task] = True
    else:
        FUSION_TRAINED[task] = False
    FUSION[task] = net
    print(f"fusion[{task}] feature widths {chans}  trained={FUSION_TRAINED[task]}")

for t in MODELS: init_fusion(t)

print("\nFeature tap layers (check these look like the last decoder feature):")
for t in MODELS:
    for k, m in MODELS[t].items():
        if m: print(f"  {t:6s} {k:6s} -> {m.tap.layer}  ({m.feat_channels} ch)  candidates: {len(m.tap.candidates)}")

## 5. Pipeline and metrics

In [ ]:
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

def load_rgb(data: bytes, max_side=MAX_SIDE, size=None):
    im = Image.open(io.BytesIO(data)).convert("RGB")
    if size is not None:
        if im.size != size: im = im.resize(size, Image.LANCZOS)
    else:
        s = max(im.size)
        if s > max_side:
            r = max_side / s; im = im.resize((max(32, round(im.width * r)), max(32, round(im.height * r))), Image.LANCZOS)
    return im

def pad16(x):
    h, w = x.shape[-2:]; ph, pw = (-h) % 16, (-w) % 16
    if ph == 0 and pw == 0: return x, (h, w)
    mode = "reflect" if (ph < h and pw < w) else "replicate"
    return F.pad(x, (0, pw, 0, ph), mode=mode), (h, w)

def to_u8(t):  # [1,3,H,W] -> HxWx3 uint8
    return (t.clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy() * 255.0).round().astype(np.uint8)

def to_y(u8):  # ITU-R BT.601 luma, as used by the Rain13k test configs (test_y_channel)
    f = u8.astype(np.float64) / 255.0
    return 65.481 * f[..., 0] + 128.553 * f[..., 1] + 24.966 * f[..., 2] + 16.0

def score(pred, gt, y_channel):
    if y_channel: pred, gt, ca = to_y(pred), to_y(gt), None
    else: pred, gt, ca = pred.astype(np.float64), gt.astype(np.float64), 2
    return float(peak_signal_noise_ratio(gt, pred, data_range=255)), float(structural_similarity(gt, pred, channel_axis=ca, data_range=255))

def data_url(u8):
    buf = io.BytesIO(); Image.fromarray(u8).save(buf, "PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

DISPLAY = {"hinet": "HINet", "sfnet": "SFNet", "adair": "AdaIR", "fusion": "Fusion (ours)"}

@torch.inference_mode()
def run_pipeline(image_bytes, task, gt_bytes=None):
    t_all = time.time()
    im = load_rgb(image_bytes)
    x0 = torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0).permute(2, 0, 1)[None].to(dev)
    x, (h, w) = pad16(x0)
    y_channel = task == "derain"
    gt_u8 = np.asarray(load_rgb(gt_bytes, size=im.size)) if gt_bytes else None

    outs, feats, items = {}, {}, []
    for key in ORDER:
        m = MODELS[task][key]; item = {"id": key, "name": DISPLAY[key], "available": False, "note": "", "image": None, "psnr": None, "ssim": None, "time_ms": None}
        if m is None:
            item["note"] = "Not loaded: " + LOAD_NOTES[task].get(key, "unknown")
        else:
            try:
                t = time.time(); torch.cuda.synchronize()
                out, feat = m(x); torch.cuda.synchronize()
                outs[key], feats[key] = out, feat
                item.update(available=True, time_ms=round((time.time() - t) * 1000))
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); item["note"] = "Out of GPU memory. Try a smaller image."
            except Exception as e:
                item["note"] = f"{type(e).__name__}: {str(e)[:160]}"
        items.append(item)

    mask = [k in outs for k in ORDER]
    fusion_item = {"id": "fusion", "name": DISPLAY["fusion"], "available": False, "note": "", "image": None, "psnr": None, "ssim": None, "time_ms": None}
    if any(mask):
        try:
            t = time.time(); torch.cuda.synchronize()
            fused, _ = FUSION[task]([outs.get(k, x) for k in ORDER], [feats.get(k) for k in ORDER], mask)
            torch.cuda.synchronize()
            outs["fusion"] = fused
            fusion_item.update(available=True, time_ms=round((time.time() - t) * 1000))
            if not FUSION_TRAINED[task]:
                fusion_item["note"] = "Placeholder: fusion weights not trained yet, so this equals a plain average."
        except Exception as e:
            fusion_item["note"] = f"{type(e).__name__}: {str(e)[:160]}"
    else:
        fusion_item["note"] = "No base model produced an output."
    items.append(fusion_item)

    for it in items:
        if it["id"] in outs:
            u8 = to_u8(outs[it["id"]][..., :h, :w])
            it["image"] = data_url(u8)
            if gt_u8 is not None: it["psnr"], it["ssim"] = score(u8, gt_u8, y_channel)
    del outs, feats; torch.cuda.empty_cache()
    return {"task": task, "width": w, "height": h, "y_channel": y_channel, "has_gt": gt_u8 is not None,
            "fusion_trained": FUSION_TRAINED[task], "elapsed_ms": round((time.time() - t_all) * 1000),
            "input": data_url(np.asarray(im)), "results": items}

## 6. API server
`GET /health`, `GET /benchmarks`, `POST /restore` (multipart: `image`, `task` = deblur|derain, optional `gt`).

In [ ]:
from typing import Optional
from fastapi import FastAPI, File, Form, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware

BENCH_FILE = f"{WORK}/benchmarks.json"
BENCH_TEMPLATE = {
    "deblur": {"dataset": "GoPro test", "n_images": None, "rows": {k: {"psnr": None, "ssim": None} for k in DISPLAY}},
    "derain": {"dataset": "Rain100L", "n_images": 100, "rows": {k: {"psnr": None, "ssim": None} for k in DISPLAY}},
}
if not os.path.exists(BENCH_FILE):
    json.dump(BENCH_TEMPLATE, open(BENCH_FILE, "w"), indent=2)   # fill with your real test-set numbers

app = FastAPI(title="Restoration Fusion API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
gpu_lock = threading.Lock()

@app.get("/health")
def health():
    return {"status": "ok", "gpu": torch.cuda.get_device_name(0), "max_side": MAX_SIDE,
            "models": {t: {k: (MODELS[t][k] is not None) for k in ORDER} for t in MODELS},
            "fusion_trained": FUSION_TRAINED}

@app.get("/benchmarks")
def benchmarks():
    return json.load(open(BENCH_FILE))

@app.post("/restore")
def restore(image: UploadFile = File(...), task: str = Form("deblur"), gt: Optional[UploadFile] = File(None)):
    if task not in MODELS: raise HTTPException(400, "task must be 'deblur' or 'derain'")
    try:
        img_bytes = image.file.read(); gt_bytes = gt.file.read() if gt is not None else None
        with gpu_lock:
            return run_pipeline(img_bytes, task, gt_bytes)
    except HTTPException: raise
    except Exception as e:
        raise HTTPException(500, f"{type(e).__name__}: {e}")

## 7. Start server and public tunnel
Copy the printed `https://....trycloudflare.com` URL into the website's "Backend URL" box.
Keep this notebook running while you demo. Quick tunnels close requests after about 100 s.

In [ ]:
import uvicorn
server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)

if not os.path.exists(f"{WORK}/cloudflared"):
    sh(f"wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {WORK}/cloudflared && chmod +x {WORK}/cloudflared")
tun = subprocess.Popen([f"{WORK}/cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for line in tun.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: public_url = m.group(0); break
print("\nBACKEND URL  ->", public_url)
threading.Thread(target=lambda: [None for _ in tun.stdout], daemon=True).start()   # keep draining the tunnel log

## 8. Optional: smoke test from inside Colab

In [ ]:
import requests
print(requests.get(f"http://localhost:{PORT}/health").json())